In [1]:
import os
import xarray as xr
import numpy as np
import pandas as pd

import fates_calibration_library.utils as utils
import fates_calibration_library.clm_functions as clm
import fates_calibration_library.surface_data_functions as surface

In [57]:
all_pfts = np.array([10.0, 40.0, 0.0, 5.0, 10.0, 0.0, 0.0, 0.0, 0.0, 0.0, 25.0, 10.0, 0.0, 0.0, 0.0, 0.0])
non_veg = all_pfts[0]
veg = all_pfts[1:]
sum_veg = veg.sum()
veg_prop = (veg/sum_veg)*100.0
sorted_pfts = sorted(veg_prop, reverse=True)
pft_sort = np.flip(np.argsort(veg_prop))

In [67]:
def get_dom_pft(dat, threshold):
    
    all_pfts = np.append(dat.PCT_NAT_PFT.values, dat.PCT_CFT.values)
    non_veg = all_pfts[0]
    veg = all_pfts[1:]
    
    if all_pfts.sum() >= threshold:
        if non_veg >= threshold:
            veg_group = 'not_vegetated'
        else:
            sum_veg = veg.sum()
            veg_prop = (veg/sum_veg)*100.0
            sorted_pfts = sorted(veg_prop, reverse=True)
            pft_sort = np.flip(np.argsort(veg_prop))
    
            if sorted_pfts[0] >= threshold:
                dom_pft_inds = [pft_sort[0].tolist()]
            else:
                cumul_prop = sorted_pfts[0]
                for i in range(1, len(sorted_pfts)):
                    cumul_prop = cumul_prop + sorted_pfts[i]
                    if cumul_prop >= threshold:
                        dom_pft_inds = pft_sort[:i+1].tolist()
                        break

            fates_dom_pft_inds = np.concatenate([clm_fates_mapping[i+1] for i in dom_pft_inds]).flatten()
            fates_dom_pfts = [pft_names[i-1] for i in fates_dom_pft_inds]
            veg_group = '-'.join(fates_dom_pfts)
    else:
        veg_group = '-'

    return veg_group

In [3]:
# surface file
surdat_dir = "/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/surfdata_esmf/ctsm5.3.0/"
surdat_2deg = os.path.join(surdat_dir, "surfdata_1.9x2.5_hist_2000_16pfts_c240908.nc")

In [4]:
# clm-fates index mapping
clm_fates_mapping_file = os.path.join("/glade/work/afoster/FATES_calibration/fates_calibration_library/configs",
                                      "clm_fates_index.yaml")
clm_fates_mapping = utils.get_config_file(clm_fates_mapping_file)

In [5]:
# pft names
param = xr.open_dataset('/glade/work/afoster/FATES_calibration/parameter_files/fates_params_default_sci.1.81.1_api.38.0.0_crops_vai.nc')
pft_names = [str(f).replace("b'", "").strip().replace(" ", "").replace("'", "") for f in param.fates_pftname.values]

In [9]:
surdat = surface.get_surdat(surdat_2deg)

In [10]:
# create a global land frac and area grid
land_frac_ds = os.path.join("/glade/derecho/scratch/afoster/archive",
                            "ctsm60SP_bigleaf_fullgrid/lnd/hist",
                            "ctsm60SP_bigleaf_fullgrid.clm2.h0.0001-02-01-00000.nc")
target_grid = clm.create_target_grid(land_frac_ds, 'FSR')
landfrac = target_grid.landfrac

In [68]:
lats = surdat.lat.values
lons = surdat.lon.values

dominant_pft = xr.DataArray(
    np.full((len(lats), len(lons)), '-', dtype='<U200'),
    dims=["lat", "lon"],
    coords=dict(
        lon=(["lon"], surdat.lon.data),
        lat=(["lat"], surdat.lat.data),
    )
)

threshold = 75

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        if landfrac.sel(lat=lat, method='nearest').sel(lon=lon,method='nearest') > 0.25:
            dat = surdat.sel(lat=lat, method='nearest').sel(lon=lon, method='nearest')
            dd = get_dom_pft(dat, threshold)
            dominant_pft[i, j] = dd
surdat['dominant_pft'] = dominant_pft

In [73]:
lut = np.unique(surdat.dominant_pft)
lut = pd.DataFrame({'dominant_pft': np.unique(surdat.dominant_pft)})
lut['ind'] = lut.index
lut.to_csv('lut.csv')

In [70]:
cat_to_int = {name: i for i, name in enumerate(np.unique(surdat.dominant_pft))}
veg_int_data = np.vectorize(cat_to_int.get)(surdat.dominant_pft.values)
veg_int = surdat.dominant_pft.copy(data=veg_int_data)